# Resultados del entrenamiento

In [ ]:
import pandas as pd
from datasets import load_dataset
import torch
from tqdm import tqdm  # Para la barra de progreso
from infer import load_model_and_tokenizer
from itables import show

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer = load_model_and_tokenizer(device=device)
model.eval()  

dataset = load_dataset("opus100", "en-es")

# Extraer las listas de textos en inglés y español
test_data = dataset['test']['translation']
en_texts = [item['en'] for item in test_data]
es_refs = [item['es'] for item in test_data]

# 3. Función de Traducción Optimizada
def translate(text):
    input_ids = tokenizer.encode(text, pad=True)
    x = torch.tensor([input_ids], dtype=torch.long).to(device)

    # CRUCIAL: Desactivar el cálculo de gradientes para acelerar y ahorrar RAM
    with torch.no_grad():
        y_pred = model.predict(
            x=x, 
            bos_id=tokenizer.start_id, 
            end_id=tokenizer.end_id, 
            device=device
        )

    translation = tokenizer.decode(y_pred[0].tolist(), skip_special_tokens=True)
    return translation

In [ ]:
model_predictions = []

# Recorremos los textos en inglés con una barra de progreso
for text in tqdm(en_texts, desc="Traduciendo", unit="frase"):
    traduccion = translate(text)
    model_predictions.append(traduccion)

# 5. Crear el DataFrame Final
df_resultados = pd.DataFrame({
    'Ingles_Origen': en_texts,
    'Espanol_Referencia': es_refs,
    'Espanol_Modelo': model_predictions
})

# Mostrar las primeras 5 filas para verificar que todo salió bien
display(df_resultados.head())

# Opcional pero recomendado: Guardar los resultados en un CSV para no perderlos
df_resultados.to_csv("resultados_opus100.csv", index=False, encoding='utf-8')

In [ ]:
import pandas as pd
from datasets import load_dataset
import torch
from tqdm import tqdm  # Para la barra de progreso
from infer import load_model_and_tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer = load_model_and_tokenizer(device=device)
model.eval()  

dataset = load_dataset("opus100", "en-es")

# Extraer las listas de textos en inglés y español
test_data = dataset['test']['translation']
en_texts = [item['en'] for item in test_data]
es_refs = [item['es'] for item in test_data]

# 3. Función de Traducción Optimizada
def translate(text):
    input_ids = tokenizer.encode(text, pad=True)
    x = torch.tensor([input_ids], dtype=torch.long).to(device)

    # CRUCIAL: Desactivar el cálculo de gradientes para acelerar y ahorrar RAM
    with torch.no_grad():
        y_pred = model.predict(
            x=x, 
            bos_id=tokenizer.start_id, 
            end_id=tokenizer.end_id, 
            device=device
        )

    translation = tokenizer.decode(y_pred[0].tolist(), skip_special_tokens=True)
    return translation

# 4. Generar Traducciones
print(f"Iniciando traducción de {len(en_texts)} oraciones...")
model_predictions = []

# Recorremos los textos en inglés con una barra de progreso
for text in tqdm(en_texts, desc="Traduciendo", unit="frase"):
    traduccion = translate(text)
    model_predictions.append(traduccion)

# 5. Crear el DataFrame Final
df_resultados = pd.DataFrame({
    'Inglés': en_texts,
    'Español': es_refs,
    'Modelo': model_predictions
})



In [17]:
show(df_resultados, 
    classes="display compact", 
    lengthMenu=[10, 10, 50],
    columnDefs=[{"className": "dt-left", "targets": "_all"}] # Aplica texto a la izquierda a todas las columnas
)

Loading ITables v2.7.3 from the internet... (need help?)
